# Phase 3: Four-Class Routing Label Design and Validation

**Project:** AI-Driven Intelligent Greywater Management and Smart Reuse Routing System  
**Objective:** Establish a scientifically defensible, multi-tier operational routing framework for greywater reuse decision-making.

> **CRITICAL SCIENTIFIC PRINCIPLE:**
> These are **operational routing decisions**, NOT claims that raw untreated greywater is safe for human contact. 
> The initial four-class routing labels are rule-derived operational labels based on documented water-quality criteria and are not equivalent to independently measured expert ground-truth labels.


In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Styling configuration
sns.set_theme(style="whitegrid")
plt.rcParams.update({'font.sans-serif': 'DejaVu Sans', 'font.size': 10})

# Add project root to sys.path
sys.path.insert(0, os.path.abspath('..'))

from routing.routing_rules import (
    evaluate_routing,
    apply_routing_to_dataframe,
    DEFAULT_THRESHOLDS,
    CLASS_SEWER_BYPASS,
    CLASS_BIO_FILTRATION,
    CLASS_RESTRICTED_IRRIGATION,
    CLASS_INDOOR_REUSE,
    CLASS_NAMES
)

print("Environment configured and routing modules loaded successfully.")


## 1. Conceptual Framework & Four Routing Classes

The system evaluates raw effluent to assign one of four operational routing destinations:

1. **CLASS 0 — SEWER BYPASS**:
   - Greywater exhibiting extreme contamination, acute pathogen spikes, toxic chemical demand, or septic anoxia that exceeds safe package treatment capabilities. Diverted immediately to municipal sewer to protect infrastructure and public health.
2. **CLASS 1 — BIO-FILTRATION**:
   - Elevated organic, particulate, or microbiological load requiring intensive biological oxidation, constructed wetlands, or sand filtration. Signifies **"TREATMENT PATHWAY MANDATED"**, not safe water.
3. **CLASS 2 — RESTRICTED IRRIGATION**:
   - Moderate-quality profile suitable for non-food crop, landscape, or turf irrigation (via sub-surface/drip application) subject to basic screen filtration and environmental context.
4. **CLASS 3 — INDOOR REUSE**:
   - Lowest-risk raw influent (typically light bathroom greywater) eligible for compact onsite packaged treatment trains (microfiltration + UV/chlorination) for non-potable toilet flushing.


## 2. Authoritative Standards and Threshold Reference

Thresholds are established using recognized international standards:
- **US EPA Guidelines for Water Reuse** (EPA 625/R-04/108 & EPA/600/R-12/618)
- **WHO Guidelines for the Safe Use of Wastewater, Excreta and Greywater** (2006, Vol 4)
- **ISO 16075-1 / ISO 16075-2** (Treated Wastewater Use for Irrigation Projects)
- **NSF/ANSI Standard 350** (Onsite Residential & Commercial Water Reuse Treatment Systems)
- **FAO Irrigation and Drainage Paper 29** (Water Quality for Agriculture)


In [ ]:
ref_path = '../reports/routing_threshold_reference.csv'
ref_df = pd.read_csv(ref_path)
print(f"Loaded {len(ref_df)} threshold rules from {ref_path}")
ref_df.head(10)


## 3. Dataset Loading (Read-Only)
Load `dataset/main.csv` in memory without modifying the original file.


In [ ]:
DATASET_PATH = '../dataset/main.csv'
raw_df = pd.read_csv(DATASET_PATH)

print(f"Dataset successfully loaded from: {DATASET_PATH}")
print(f"Observations (Rows): {raw_df.shape[0]}")
print(f"Attributes (Columns): {raw_df.shape[1]}")


## 4. Applying Multi-Tier Hierarchical Decision Logic
Execute `apply_routing_to_dataframe` across all 1,500 samples.


In [ ]:
labeled_df = apply_routing_to_dataframe(raw_df)

num_cols = [
    'pH', 'TEMP_C', 'SAL_ppt', 'TUR_NTU', 'DS_mg_L', 'TDS_mg_L',
    'TSS_mg_L', 'COND_uS_cm', 'DO_mg_L', 'BOD_mg_L', 'COD_mg_L',
    'NH4F_mg_L', 'NO3_mg_L', 'K_mg_L', 'E_coli_CFU_100mL'
]

export_cols = [
    'Sample_ID', 'Greywater_Source'
] + num_cols + [
    'Routing_Class', 'Routing_Class_Name', 'Primary_Routing_Reason',
    'Triggered_Parameters', 'Treatment_Required', 'Safety_Flag'
]

final_export_df = labeled_df[export_cols]

# Save derived dataset
PROCESSED_PATH = '../dataset/processed/greywater_routing_labels.csv'
final_export_df.to_csv(PROCESSED_PATH, index=False)
print(f"Successfully generated derived routing dataset: {PROCESSED_PATH}")
final_export_df.head()


## 5. Routing Class Distribution Analysis
Analyze the sample frequencies across the four operational classes.


In [ ]:
dist_counts = final_export_df['Routing_Class_Name'].value_counts()
dist_pcts = (final_export_df['Routing_Class_Name'].value_counts(normalize=True) * 100).round(2)
dist_df = pd.DataFrame({'Sample_Count': dist_counts, 'Percentage (%)': dist_pcts})
print("Four-Class Routing Distribution:")
dist_df


In [ ]:
plt.figure(figsize=(9, 5.5))
palette = ['#e63946', '#f4a261', '#2a9d8f', '#264653']
order = ['Sewer Bypass', 'Bio-filtration', 'Restricted Irrigation', 'Indoor Reuse']

ax = sns.barplot(
    data=final_export_df,
    x='Routing_Class_Name',
    order=order,
    palette=palette,
    hue='Routing_Class_Name',
    legend=False
)
plt.title('Four-Class Greywater Reuse Routing Distribution (Baseline Rules)', fontsize=13, weight='bold', pad=14)
plt.xlabel('Operational Routing Decision', fontsize=11, weight='bold')
plt.ylabel('Number of Samples', fontsize=11, weight='bold')

for p in ax.patches:
    h = p.get_height()
    pct = (h / len(final_export_df)) * 100
    ax.annotate(f"{int(h)}\n({pct:.1f}%)",
                (p.get_x() + p.get_width() / 2., h / 2 if h > 30 else h + 15),
                ha='center', va='center', fontsize=10,
                color='white' if h > 30 else 'black', weight='bold')

plt.tight_layout()
plt.savefig('../reports/figures/routing_class_distribution.png', dpi=300)
plt.show()


## 6. Source-Wise Routing Analysis
Evaluate how effluent origin governs routing decisions.


In [ ]:
source_summary = pd.crosstab(
    final_export_df['Greywater_Source'],
    final_export_df['Routing_Class_Name']
)[order]
source_summary['Total'] = source_summary.sum(axis=1)
print("Source-Wise Routing Breakdown:")
source_summary


In [ ]:
plot_xtab = pd.crosstab(
    final_export_df['Greywater_Source'],
    final_export_df['Routing_Class_Name'],
    normalize='index'
)[order] * 100

ax = plot_xtab.plot(
    kind='bar',
    stacked=True,
    figsize=(10, 6),
    color=palette,
    edgecolor='white',
    linewidth=0.8
)
plt.title('Source-Wise Composition of Multi-Tier Routing Decisions', fontsize=13, weight='bold', pad=14)
plt.xlabel('Greywater Source', fontsize=11, weight='bold')
plt.ylabel('Percentage within Source (%)', fontsize=11, weight='bold')
plt.legend(title='Routing Decision', frameon=True, facecolor='white', framealpha=0.9)
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig('../reports/figures/routing_class_by_source.png', dpi=300)
plt.show()


## 7. Label Consistency & Integrity Verification
Conduct rigorous sanity checks on generated routing labels:
- Missing or out-of-range class codes
- Unjustified Sewer Bypass decisions
- Inappropriate Indoor Reuse classifications with elevated contaminants
- Priority conflict masking (low TDS masking high BOD/pathogen loads)


In [ ]:
validation_path = '../reports/routing_label_validation.csv'
val_df = pd.read_csv(validation_path)
val_df


## 8. Threshold Sensitivity Analysis
Evaluate class distribution stability across Stringent, Baseline, and Lenient threshold envelopes.


In [ ]:
sens_path = '../reports/routing_sensitivity_analysis.csv'
sens_df = pd.read_csv(sens_path)
sens_df


## 9. Scientific Limitations & Key Findings

1. **Natural Scarcity of Raw Indoor Reuse Samples:**
   - In raw untreated greywater, only **6 samples (0.40%)** qualify for the `Indoor Reuse` influent envelope under baseline rules (and 0% under stringent rules).
   - This occurs because raw greywater has high baseline pathogen and organic loads (*E. coli* minimum across 1,500 samples is 1,649 CFU/100mL; median is 60,278 CFU/100mL; median BOD is 170.4 mg/L).
   - **Crucial Takeaway:** Untreated domestic greywater cannot be directly reused indoors; it requires multi-stage filtration and disinfection.
2. **Rule-Derived Operational Labels:**
   - The labels generated in this phase reflect regulatory screening logic, not experimentally observed post-treatment efficacy.
   - These labels establish the ground truth for training supervised classification models (XGBoost / Random Forest) in Phase 4.
